In [1]:
!pip install mlflow dagshub dvc fastparquet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 75.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [24]:
import pandas as pd
import numpy as np
import dagshub
import mlflow
from getpass import getpass
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

In [3]:
!git clone https://github.com/MuhammadShaafImran/Real-Time-Market-Movement-Prediction-System.git
%cd Real-Time-Market-Movement-Prediction-System

Cloning into 'Real-Time-Market-Movement-Prediction-System'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 74 (delta 24), reused 64 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 215.51 KiB | 19.59 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Real-Time-Market-Movement-Prediction-System


In [4]:
!ls

LICENSE  README.md


In [5]:
!git checkout dev
!ls

Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
LICENSE  README.md  requirements.txt  src


In [6]:
DAGSHUB_USER = "RudhanLodhi"
print("Enter  DagsHub key:")
DAGSHUB_TOKEN = getpass().strip()

os.system('dvc remote modify origin --local auth basic')
os.system(f'dvc remote modify origin --local user {DAGSHUB_USER}')
os.system(f'dvc remote modify origin --local password {DAGSHUB_TOKEN}')

exit_code = os.system('dvc pull')

if exit_code == 0:
    print("✅ Success! Your images have been downloaded.")
else:
    print("❌ Error: dvc pull failed. ")

Enter  DagsHub key:
✅ Success! Your images have been downloaded.


In [7]:
!ls src/data

processed  processed.dvc  raw  raw.dvc


In [8]:
dataset_path = 'src/data/processed/latest_ml_dataset_v4_finbert.parquet'
df = pd.read_parquet(dataset_path)
df.head()

,timestamp,symbol,open,high,low,close,volume,RSI,MACD,MACD_signal,SMA_20,EMA_20,BB_high,BB_low,ticker_sentiment,news_count,market_sentiment,reddit_hype,label
0,2026-02-13 14:30:00+00:00,AAPL,262.010010,262.230011,258.799988,259.299988,4578357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
1,2026-02-13 14:35:00+00:00,AAPL,259.339996,260.260010,258.799988,259.385010,772714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
2,2026-02-13 14:40:00+00:00,AAPL,259.350006,260.190002,259.019989,259.519989,646951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,1
3,2026-02-13 14:45:00+00:00,AAPL,259.510010,260.980011,259.399994,260.274994,611692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0
4,2026-02-13 14:50:00+00:00,AAPL,260.269989,260.326691,258.820007,259.500000,564613,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0


In [9]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

Dataset shape: (14037, 19)

Columns: ['timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'SMA_20', 'EMA_20', 'BB_high', 'BB_low', 'ticker_sentiment', 'news_count', 'market_sentiment', 'reddit_hype', 'label']


In [10]:
print(f"\nData types:")
print(df.dtypes)


Data types:
timestamp           datetime64[ns, UTC]
symbol                           object
open                            float64
high                            float64
low                             float64
close                           float64
volume                            int64
RSI                             float64
MACD                            float64
MACD_signal                     float64
SMA_20                          float64
EMA_20                          float64
BB_high                         float64
BB_low                          float64
ticker_sentiment                float64
news_count                      float64
market_sentiment                float64
reddit_hype                     float64
label                             int32
dtype: object


In [11]:
print(f"\nMissing values:")
print(df.isnull().sum().to_frame("missing").query("missing > 0"))
print(f"\nTarget distribution:")
print(df['label'].value_counts())


Missing values:
             missing
RSI               39
MACD              75
MACD_signal       99
SMA_20            57
EMA_20            57
BB_high           57
BB_low            57

Target distribution:
label
0    7019
1    7018
Name: count, dtype: int64


## Preprocessing

In [12]:
df_sorted = df.sort_values('timestamp').reset_index(drop=True)

y = df_sorted['label'].values
X = df_sorted.drop(['label', 'timestamp'], axis=1)

categorical_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {numeric_cols}")

Categorical columns: ['symbol']
Numeric columns: ['open', 'high', 'low', 'close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'SMA_20', 'EMA_20', 'BB_high', 'BB_low', 'ticker_sentiment', 'news_count', 'market_sentiment', 'reddit_hype']


In [13]:
X_processed = X.copy()

for col in numeric_cols:
    if X_processed[col].isnull().sum() > 0:
        X_processed[col] = X_processed[col].ffill().bfill().fillna(X_processed[col].mean())

for col in categorical_cols:
    X_processed[col] = X_processed[col].fillna("Unknown").astype(str)
    X_processed[col] = LabelEncoder().fit_transform(X_processed[col])

y_raw = y
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print(f"\nProcessed features shape: {X_processed.shape}")
print(f"Missing values after preprocessing: {X_processed.isnull().sum().sum()}")
print(f"Target shape: {y.shape}")
print(f"\nFeature statistics:")
print(X_processed.describe())


Processed features shape: (14037, 17)
Missing values after preprocessing: 0
Target shape: (14037,)

Feature statistics:
             symbol          open          high           low         close  \
count  14037.000000  14037.000000  14037.000000  14037.000000  14037.000000   
mean       1.000000    324.562520    324.987535    324.137906    324.572444   
std        0.816526     55.719987     55.871132     55.566657     55.727084   
min        0.000000    245.509995    245.949997    245.509995    245.529999   
25%        0.000000    270.609985    270.915009    270.399994    270.619995   
50%        1.000000    308.541687    308.859985    308.172699    308.540009   
75%        2.000000    382.489990    383.100006    381.850006    382.529999   
max        2.000000    447.989990    449.160004    447.519989    447.980011   

             volume           RSI          MACD   MACD_signal        SMA_20  \
count  1.403700e+04  14037.000000  14037.000000  14037.000000  14037.000000   
mean   5.

In [14]:
n_samples = len(X_processed)
train_size = int(0.7 * n_samples)  # 70% for training
val_size = int(0.1 * n_samples)    # 10% for validation
test_size = n_samples - train_size - val_size  # 20% for testing

# Split chronologically
X_train = X_processed.iloc[:train_size].values
y_train = y[:train_size]

X_val = X_processed.iloc[train_size:train_size + val_size].values
y_val = y[train_size:train_size + val_size]

X_test = X_processed.iloc[train_size + val_size:].values
y_test = y[train_size + val_size:]

print(f"Train set: {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")
print(f"\nClass distribution in train: {np.bincount(y_train)}")
print(f"Class distribution in val: {np.bincount(y_val)}")
print(f"Class distribution in test: {np.bincount(y_test)}")


Train set: (9825, 17), (9825,)
Validation set: (1403, 17), (1403,)
Test set: (2809, 17), (2809,)

Class distribution in train: [4919 4906]
Class distribution in val: [719 684]
Class distribution in test: [1381 1428]


In [15]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
feature_names = X_processed.columns.tolist()

print(f"\nFeatures scaled successfully")
print(f"Train mean: {X_train_scaled.mean(axis=0)[:5]}, std: {X_train_scaled.std(axis=0)[:5]}")
print(f"Validation shape: {X_val_scaled.shape}, Test shape: {X_test_scaled.shape}")


Features scaled successfully
Train mean: [0.5        0.39758287 0.39742128 0.39687142 0.39754442], std: [0.40824829 0.30896935 0.30975285 0.30940698 0.30901628]
Validation shape: (1403, 17), Test shape: (2809, 17)


In [16]:
def create_sequences(X, y, seq_length=30):
    """
    Create sequences for time-series models.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        y: Target labels (n_samples,)
        seq_length: Length of each sequence
    
    Returns:
        X_seq: Sequences (n_sequences, seq_length, n_features)
        y_seq: Corresponding labels (n_sequences,)
    """
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        # Use the label of the next time step as target
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

SEQ_LENGTH = 30  # Use 30 previous time steps to predict the next

# Generate sequences
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, SEQ_LENGTH)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val, SEQ_LENGTH)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, SEQ_LENGTH)

print(f"Sequence generation completed with sequence length: {SEQ_LENGTH}")
print(f"Train sequences: {X_train_seq.shape}, labels: {y_train_seq.shape}")
print(f"Val sequences: {X_val_seq.shape}, labels: {y_val_seq.shape}")
print(f"Test sequences: {X_test_seq.shape}, labels: {y_test_seq.shape}")
print(f"Input features per timestep: {X_train_seq.shape[2]}")

Sequence generation completed with sequence length: 30
Train sequences: (9795, 30, 17), labels: (9795,)
Val sequences: (1373, 30, 17), labels: (1373,)
Test sequences: (2779, 30, 17), labels: (2779,)
Input features per timestep: 17


In [17]:
X_train_tensor = torch.FloatTensor(X_train_seq)
y_train_tensor = torch.LongTensor(y_train_seq)

X_val_tensor = torch.FloatTensor(X_val_seq)
y_val_tensor = torch.LongTensor(y_val_seq)

X_test_tensor = torch.FloatTensor(X_test_seq)
y_test_tensor = torch.LongTensor(y_test_seq)

print(f"\nTensors created successfully")
print(f"Device availability - CUDA: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Tensors created successfully
Device availability - CUDA: True
Using device: cuda


## Training

In [19]:
class RNNModel(nn.Module):
    """Simple RNN model for binary classification"""
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, 
                          batch_first=True, dropout=dropout)
        
        # Upgraded to a Dense Block for better generalization
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 2)  # 2 classes (raw logits)
        )
        
    def forward(self, x):
        _, h_n = self.rnn(x)
        out = self.classifier(h_n[-1])
        return out

class GRUModel(nn.Module):
    """GRU model for binary classification"""
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.gru = nn.GRU(input_size, hidden_size, num_layers, 
                          batch_first=True, dropout=dropout)
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
        
    def forward(self, x):
        _, h_n = self.gru(x)
        out = self.classifier(h_n[-1])
        return out

class LSTMModel(nn.Module):
    """LSTM model for binary classification"""
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout)
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
        
    def forward(self, x):
        _, (h_n, c_n) = self.lstm(x)
        out = self.classifier(h_n[-1])
        return out

In [20]:
INPUT_SIZE = X_train_seq.shape[2]
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

print(f"Model configuration:")
print(f"- Input size: {INPUT_SIZE}")
print(f"- Hidden size: {HIDDEN_SIZE}")
print(f"- Number of layers: {NUM_LAYERS}")
print(f"- Dropout: {DROPOUT}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Epochs: {EPOCHS}")
print(f"- Learning rate: {LEARNING_RATE}")

Model configuration:
- Input size: 17
- Hidden size: 64
- Number of layers: 2
- Dropout: 0.2
- Batch size: 32
- Epochs: 50
- Learning rate: 0.001


## Set tracking

In [21]:
# https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.dvc
dagshub.init(repo_owner='shaafimran257', repo_name='Real-Time-Market-Movement-Prediction-System', mlflow=True)
mlflow.set_experiment("Market_Movement_Analysis")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=05eac563-21b4-415f-929a-48d710b2014d&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1c9442bf54dc04ca921446a1c72ecac6a903439220694e9ddb79e99fe48f9eb1




Accessing as RudhanLodhi

Initialized MLflow to track repo "shaafimran257/Real-Time-Market-Movement-Prediction-System"

Repository shaafimran257/Real-Time-Market-Movement-Prediction-System initialized!

MLflow tracking URI: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow


In [22]:
DAGSHUB_USER = "shaafimran257"  
DAGSHUB_REPO = "Real-Time-Market-Movement-Prediction-System"

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("Market-Movement-RNN-GRU-LSTM")

print("MLflow configured successfully")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('Market-Movement-RNN-GRU-LSTM').experiment_id}")

MLflow configured successfully
Tracking URI: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow
Experiment: 1


In [23]:
def calculate_accuracy(y_true, y_pred_classes):
    """Calculate Accuracy for binary classification"""
    return accuracy_score(y_true, y_pred_classes)

def calculate_f1(y_true, y_pred_classes):
    """Calculate F1-Score for binary classification"""
    # Use macro or weighted if classes are imbalanced; binary is default
    return f1_score(y_true, y_pred_classes, average='weighted')

def calculate_rmse(y_true, y_pred):
    """
    Calculate RMSE. 
    Note: Only use this if you change your models to predict raw continuous prices 
    (regression) instead of direction (classification).
    """
    mse = np.mean((y_true - y_pred) ** 2)
    return np.sqrt(mse)

In [25]:
def train_and_evaluate_model(model_class, model_name, X_train, y_train, X_val, y_val, X_test, y_test):
    
    # Start MLflow run
    with mlflow.start_run(run_name=model_name):
        # Initialize model
        model = model_class(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
        model = model.to(device)
        
        # Loss function and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
        # Log hyperparameters
        mlflow.log_params({
            'model_type': model_name,
            'input_size': INPUT_SIZE,
            'hidden_size': HIDDEN_SIZE,
            'num_layers': NUM_LAYERS,
            'dropout': DROPOUT,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS,
            'sequence_length': SEQ_LENGTH,
            'optimizer': 'Adam',
            'loss_function': 'CrossEntropyLoss'
        })
        
        # Create data loaders
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        
        best_val_accuracy = 0
        best_model_state = None
        
        for epoch in range(EPOCHS):
            # Training phase
            model.train()
            train_loss = 0
            train_correct = 0
            train_total = 0
            
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                # Forward pass
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                train_correct += (predicted == y_batch).sum().item()
                train_total += y_batch.size(0)
            
            train_loss /= len(train_loader)
            train_accuracy = train_correct / train_total
            
            # Validation phase
            model.eval()
            with torch.no_grad():
                val_outputs = model(X_val.to(device))
                val_loss = criterion(val_outputs, y_val.to(device)).item()
                _, val_predicted = torch.max(val_outputs, 1)
                val_correct = (val_predicted == y_val.to(device)).sum().item()
                val_accuracy = val_correct / len(y_val)
            
            # Save best model
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                best_model_state = model.state_dict().copy()
            
            # Log metrics to MLflow
            mlflow.log_metric('train_loss', train_loss, step=epoch)
            mlflow.log_metric('val_loss', val_loss, step=epoch)
            mlflow.log_metric('train_accuracy', train_accuracy, step=epoch)
            mlflow.log_metric('val_accuracy', val_accuracy, step=epoch)
            
            if (epoch + 1) % 5 == 0:
                print(f"[{model_name}] Epoch {epoch+1}/{EPOCHS} | Train Acc: {train_accuracy:.4f} | Val Acc: {val_accuracy:.4f}")
        
        # Load best weights
        model.load_state_dict(best_model_state)
        
        # Test phase
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test.to(device))
            test_loss = criterion(test_outputs, y_test.to(device)).item()
            _, test_predicted = torch.max(test_outputs, 1)
            
            y_true_np = y_test.cpu().numpy()
            y_pred_np = test_predicted.cpu().numpy()
            
            test_accuracy = (test_predicted == y_test.to(device)).sum().item() / len(y_test)
            test_f1 = f1_score(y_true_np, y_pred_np, average='weighted') # 'weighted' handles potential imbalance
            test_rmse = calculate_rmse(y_true_np, y_pred_np)
            cm = confusion_matrix(y_true_np, y_pred_np)
        
        # Final MLflow logs
        mlflow.log_metrics({
            'final_test_accuracy': test_accuracy,
            'final_test_f1_score': test_f1,
            'final_test_rmse': test_rmse
        })
        
        # Save Model Artifact
        mlflow.pytorch.log_model(model, f"{model_name}_model")
        
        # Confusion Matrix Plot
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'CM: {model_name}')
        plt.ylabel('True')
        plt.xlabel('Pred')
        cm_path = f"{model_name}_cm.png"
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()
        
        return model, best_val_accuracy, test_accuracy

In [26]:
X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)
X_val_tensor = X_val_tensor.to(device)
y_val_tensor = y_val_tensor.to(device)
X_test_tensor = X_test_tensor.to(device)
y_test_tensor = y_test_tensor.to(device)

results = {}

In [27]:
# --- Train RNN Model ---
print("\n" + "="*60)
print("Training RNN Model...")
print("="*60)
rnn_model, rnn_val_acc, rnn_metrics = train_and_evaluate_model(
    RNNModel, "RNN", X_train_tensor, y_train_tensor, 
    X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor
)
results['RNN'] = rnn_metrics


Training RNN Model...
[RNN] Epoch 5/50 | Train Acc: 0.4962 | Val Acc: 0.4902
[RNN] Epoch 10/50 | Train Acc: 0.4923 | Val Acc: 0.4894
[RNN] Epoch 15/50 | Train Acc: 0.4960 | Val Acc: 0.4902
[RNN] Epoch 20/50 | Train Acc: 0.4896 | Val Acc: 0.5098
[RNN] Epoch 25/50 | Train Acc: 0.4941 | Val Acc: 0.4902
[RNN] Epoch 30/50 | Train Acc: 0.4955 | Val Acc: 0.5098
[RNN] Epoch 35/50 | Train Acc: 0.4977 | Val Acc: 0.4902
[RNN] Epoch 40/50 | Train Acc: 0.4943 | Val Acc: 0.4902
[RNN] Epoch 45/50 | Train Acc: 0.4957 | Val Acc: 0.5098
[RNN] Epoch 50/50 | Train Acc: 0.4932 | Val Acc: 0.5098


2026/05/12 14:44:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 14:44:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/12 14:44:51 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 14:45:02 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label

🏃 View run RNN at: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow/#/experiments/1/runs/f940c3653e5d4f65b82be2b79094777c
🧪 View experiment at: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow/#/experiments/1


In [28]:
# --- Train GRU Model ---
print("\n" + "="*60)
print("Training GRU Model...")
print("="*60)
gru_model, gru_val_acc, gru_metrics = train_and_evaluate_model(
    GRUModel, "GRU", X_train_tensor, y_train_tensor, 
    X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor
)
results['GRU'] = gru_metrics


Training GRU Model...
[GRU] Epoch 5/50 | Train Acc: 0.5007 | Val Acc: 0.4902
[GRU] Epoch 10/50 | Train Acc: 0.4998 | Val Acc: 0.5098
[GRU] Epoch 15/50 | Train Acc: 0.4981 | Val Acc: 0.5098
[GRU] Epoch 20/50 | Train Acc: 0.4977 | Val Acc: 0.5098
[GRU] Epoch 25/50 | Train Acc: 0.4987 | Val Acc: 0.5098
[GRU] Epoch 30/50 | Train Acc: 0.5010 | Val Acc: 0.4902
[GRU] Epoch 35/50 | Train Acc: 0.4977 | Val Acc: 0.4902
[GRU] Epoch 40/50 | Train Acc: 0.4987 | Val Acc: 0.5113
[GRU] Epoch 45/50 | Train Acc: 0.4984 | Val Acc: 0.5135
[GRU] Epoch 50/50 | Train Acc: 0.5006 | Val Acc: 0.5098


2026/05/12 14:48:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 14:48:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/12 14:48:42 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 14:48:47 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label

🏃 View run GRU at: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow/#/experiments/1/runs/03a9f62e1b954019b3cb4d86cbd07833
🧪 View experiment at: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow/#/experiments/1


In [29]:
print("\n" + "="*60)
print("Training LSTM Model...")
print("="*60)
lstm_model, lstm_val_acc, lstm_metrics = train_and_evaluate_model(
    LSTMModel, "LSTM", X_train_tensor, y_train_tensor, 
    X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor
)
results['LSTM'] = lstm_metrics


Training LSTM Model...
[LSTM] Epoch 5/50 | Train Acc: 0.4940 | Val Acc: 0.5098
[LSTM] Epoch 10/50 | Train Acc: 0.5069 | Val Acc: 0.4902
[LSTM] Epoch 15/50 | Train Acc: 0.5034 | Val Acc: 0.4902
[LSTM] Epoch 20/50 | Train Acc: 0.4969 | Val Acc: 0.5098
[LSTM] Epoch 25/50 | Train Acc: 0.5017 | Val Acc: 0.4902
[LSTM] Epoch 30/50 | Train Acc: 0.4966 | Val Acc: 0.4902
[LSTM] Epoch 35/50 | Train Acc: 0.5048 | Val Acc: 0.4902
[LSTM] Epoch 40/50 | Train Acc: 0.5214 | Val Acc: 0.4945
[LSTM] Epoch 45/50 | Train Acc: 0.4986 | Val Acc: 0.4924
[LSTM] Epoch 50/50 | Train Acc: 0.5184 | Val Acc: 0.5135


2026/05/12 14:53:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 14:53:22 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/12 14:53:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 14:53:28 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu128) contains a local version label

🏃 View run LSTM at: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow/#/experiments/1/runs/68ce53e468a249b9bb73cd97cd33132a
🧪 View experiment at: https://dagshub.com/shaafimran257/Real-Time-Market-Movement-Prediction-System.mlflow/#/experiments/1


### Trying with different Hyperparams